--------------------------------------------------------------------------------
1585 has width=10000, but 1870 has width=2000.
1585 has add_bc=False, but 1870 has add_bc=True.
1585 has do_masking=False, but 1870 has do_masking=True.
1585 has train_maxes=[0.00, 1.00, 1.00, 1.00], but 1870 has train_maxes=[1.00, 1.00, 1.00, 1.00].
1585 has val_mins=[0.00, 0.20, 0.20, 0.20], but 1870 has val_mins=[0.00, 0.00, 0.00, 0.00].
1585 has val_maxes=[0.10, 0.80, 0.80, 0.80], but 1870 has val_maxes=[1.00, 1.00, 1.00, 1.00].
1585 has soln=1, but 1870 has soln=4.
1585 has param_count=200000, but 1870 has param_count=40000.
1585 has mu_inputs=[0.00, 0.52, 0.46, 0.44], but 1870 has mu_inputs=[0.42, 0.49, 0.46, 0.48].
1585 has sigma_inputs=[0.00, 0.29, 0.28, 0.30], but 1870 has sigma_inputs=[0.32, 0.35, 0.37, 0.37].
1585 has mu_targets=[-0.05, 0.02, -0.01, 0.04, 0.02, -0.01], but 1870 has mu_targets=[-0.03, 0.03, 0.06, 0.08, -0.05, 0.06].
1585 has sigma_targets=[0.26, 0.20, 0.26, 0.25, 0.22, 0.26], but 1870 has sigma_targets=[0.11, 0.13, 0.16, 0.10, 0.13, 0.16].
num_ps in 1870 but not in 1585.
q in 1870 but not in 1585.
b in 1870 but not in 1585.
init_lr in 1870 but not in 1585.
init_wd in 1870 but not in 1585.
init_max_epochs in 1870 but not in 1585.
f.py appears different.
--------------------------------------------------------------------------------

In [33]:
import os
import json

import numpy as np
import pandas as pd

In [34]:
runs = [
    os.path.join('experiments', str(x)) for x in range(1870, 1905)
]

assert len(runs) == 35, len(runs)

In [35]:
soln_number_to_name = {
    1: 'Plane waves',
    2: 'Radial waves',
    3: 'Hopf fibration',
    4: 'Random solution'
}

In [36]:
df = pd.DataFrame(
    columns=list(json.load(open(os.path.join(runs[0], 'hparams.json'))).keys())
    + [
        'soln_name',
        'min_mse_val',
        'time_to_min_mse_val',
        'min_rel_l2_error',
        'time_to_min_rel_l2_error',
        'exp_num',
        # 'time_to_5p_rel_l2_error'
    ]
)

for run in runs:
    info = json.load(open(os.path.join(run, 'hparams.json')))
    
    info['exp_num'] = int(run.split('/')[1])

    assert (info['do_masking'] and info['add_bc']) or (not info['add_bc'])
    
    info['soln_name'] = soln_number_to_name[info['soln']]
    
    log = pd.read_csv(os.path.join(run, 'log.tsv'), sep='\t')
    
    # print(log['training_time'].cumsum().max())
    
    if os.path.exists(os.path.join(run, 'init_time.txt')):
        init_time = float(open(os.path.join(run, 'init_time.txt')).read())
    else:
        init_time = 0.
    
    # epoch where we reach minimum validation MSE
    min_mse_val = log['mse_val'].min()
    info['min_mse_val'] = min_mse_val
    
    min_mse_val_epoch = int(log[log['mse_val'] == min_mse_val].iloc[0]['epoch'])
    # how much training time to get there?
    info['time_to_min_mse_val'] = log[log['epoch'] <= min_mse_val_epoch]['training_time'].sum().item() + init_time
    
    # epoch where we reach minimum validation relative L2 error
    min_rel_l2_error = log['rel_l2_error'].min()
    info['min_rel_l2_error'] = min_rel_l2_error
    
    min_rel_l2_error_epoch = int(log[log['rel_l2_error'] <= min_rel_l2_error].iloc[0]['epoch'])
    info['time_to_min_rel_l2_error'] = log[log['epoch'] <= min_rel_l2_error_epoch]['training_time'].sum().item() + init_time
    
    # epoch where we first reach 5% relative error
    # first_5p_epoch = int(log[log['rel_l2_error'] < 0.05].iloc[0]['epoch'])
    # info['time_to_5p_rel_l2_error'] = log[log['epoch'] <= first_5p_epoch]['training_time'].sum().item() + init_time
    
    df.loc[len(df)] = info

In [37]:
df

,num_ps,input_dim,output_dim,max_epochs,max_time,log_freq,verbose,restart_patience,restart_tolerance,restarts,...,mu_inputs,sigma_inputs,mu_targets,sigma_targets,soln_name,min_mse_val,time_to_min_mse_val,min_rel_l2_error,time_to_min_rel_l2_error,exp_num
0,2,3,6,10000,10,100,False,10,0.1,False,...,"[0.42362719774246216, 0.4947110712528229, 0.45...","[0.3202417194843292, 0.35279783606529236, 0.37...","[-0.03345438838005066, 0.030121732503175735, 0...","[0.11443682014942169, 0.13271602988243103, 0.1...",Random solution,0.004024,5.051832,0.443758,5.051832,1870
1,2,3,6,10000,10,100,False,10,0.1,False,...,"[0.42362719774246216, 0.4947110712528229, 0.45...","[0.3202417194843292, 0.35279783606529236, 0.37...","[-0.03345438838005066, 0.030121732503175735, 0...","[0.11443682014942169, 0.13271602988243103, 0.1...",Random solution,0.005183,23.710198,0.503609,23.710198,1871
2,2,3,6,10000,10,100,False,10,0.1,False,...,"[0.42362719774246216, 0.4947110712528229, 0.45...","[0.3202417194843292, 0.35279783606529236, 0.37...","[-0.03345438838005066, 0.030121732503175735, 0...","[0.11443682014942169, 0.13271602988243103, 0.1...",Random solution,0.005928,151.245075,0.538580,151.245075,1872
3,2,3,6,10000,10,100,False,10,0.1,False,...,"[0.42362719774246216, 0.4947110712528229, 0.45...","[0.3202417194843292, 0.35279783606529236, 0.37...","[-0.03345438838005066, 0.030121732503175735, 0...","[0.11443682014942169, 0.13271602988243103, 0.1...",Random solution,0.005251,394.248103,0.506907,394.248103,1873
4,2,3,6,10000,10,100,False,10,0.1,False,...,"[0.42362719774246216, 0.4947110712528229, 0.45...","[0.3202417194843292, 0.35279783606529236, 0.37...","[-0.03345438838005066, 0.030121732503175735, 0...","[0.11443682014942169, 0.13271602988243103, 0.1...",Random solution,0.009057,3.285520,0.665738,3.285520,1874
5,2,3,6,10000,10,100,False,10,0.1,False,...,"[0.42362719774246216, 0.4947110712528229, 0.45...","[0.3202417194843292, 0.35279783606529236, 0.37...","[-0.03345438838005066, 0.030121732503175735, 0...","[0.11443682014942169, 0.13271602988243103, 0.1...",Random solution,0.014166,4.959482,0.832589,4.959482,1875
6,2,3,6,10000,10,100,False,10,0.1,False,...,"[0.42362719774246216, 0.4947110712528229, 0.45...","[0.3202417194843292, 0.35279783606529236, 0.37...","[-0.03345438838005066, 0.030121732503175735, 0...","[0.11443682014942169, 0.13271602988243103, 0.1...",Random solution,0.017220,12.444210,0.917942,12.444210,1876
7,2,3,6,10000,10,100,False,10,0.1,False,...,"[0.41867607831954956, 0.5020413398742676, 0.49...","[0.3178815543651581, 0.35919785499572754, 0.35...","[-0.043669361621141434, 0.04419444501399994, 0...","[0.1148931235074997, 0.14607803523540497, 0.16...",Random solution,0.000050,99.771448,0.050022,99.771448,1877
8,2,3,6,10000,10,100,False,10,0.1,False,...,"[0.41867607831954956, 0.5020413398742676, 0.49...","[0.3178815543651581, 0.35919785499572754, 0.35...","[-0.043669361621141434, 0.04419444501399994, 0...","[0.1148931235074997, 0.14607803523540497, 0.16...",Random solution,0.000050,124.360106,0.050003,124.360106,1878
9,2,3,6,10000,10,100,False,10,0.1,False,...,"[0.41867607831954956, 0.5020413398742676, 0.49...","[0.3178815543651581, 0.35919785499572754, 0.35...","[-0.043669361621141434, 0.04419444501399994, 0...","[0.1148931235074997, 0.14607803523540497, 0.16...",Random solution,0.000051,89.243108,0.050177,89.243108,1879


In [38]:
df['min_rel_l2_error'] = df['min_rel_l2_error'] * 100

df = df.round(1)

for col in df.columns:
    df[col] = df[col].astype(str)

In [39]:
results = df[[
    'exp_num',
    'soln_name',
    'width',
    'activation',
    'n_train',
    # 'time_to_5p_rel_l2_error',
    'min_rel_l2_error',
    'time_to_min_rel_l2_error',
]].rename(columns={
    'exp_num': 'Experiment #',
    # 'time_to_5p_rel_l2_error': 'Time to <5% RL2E. (s)',
    'time_to_min_rel_l2_error': 'Time to min. RL2E. (s)',
    'min_rel_l2_error': 'Min. RL2E. (%)',
    'soln_name': 'Solution',
    'width': 'Network width',
    'activation': 'Activation',
    'n_train': '# points'
})

results

,Experiment #,Solution,Network width,Activation,# points,Min. RL2E. (%),Time to min. RL2E. (s)
0,1870,Random solution,2000,tanh,100,44.4,5.1
1,1871,Random solution,4000,tanh,100,50.4,23.7
2,1872,Random solution,8000,tanh,100,53.9,151.2
3,1873,Random solution,16000,tanh,100,50.7,394.2
4,1874,Random solution,32000,tanh,100,66.6,3.3
5,1875,Random solution,64000,tanh,100,83.3,5.0
6,1876,Random solution,128000,tanh,100,91.8,12.4
7,1877,Random solution,2000,tanh,1000,5.0,99.8
8,1878,Random solution,4000,tanh,1000,5.0,124.4
9,1879,Random solution,8000,tanh,1000,5.0,89.2


In [40]:
print(results.to_latex())

\begin{tabular}{llllllll}
\toprule
{} & Experiment \# &         Solution & Network width & Activation & \# points & Min. RL2E. (\%) & Time to min. RL2E. (s) \\
\midrule
0  &         1870 &  Random solution &          2000 &       tanh &      100 &           44.4 &                    5.1 \\
1  &         1871 &  Random solution &          4000 &       tanh &      100 &           50.4 &                   23.7 \\
2  &         1872 &  Random solution &          8000 &       tanh &      100 &           53.9 &                  151.2 \\
3  &         1873 &  Random solution &         16000 &       tanh &      100 &           50.7 &                  394.2 \\
4  &         1874 &  Random solution &         32000 &       tanh &      100 &           66.6 &                    3.3 \\
5  &         1875 &  Random solution &         64000 &       tanh &      100 &           83.3 &                    5.0 \\
6  &         1876 &  Random solution &        128000 &       tanh &      100 &           91.8 &    

/tmp/ipykernel_9273/3863709051.py:1: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  print(results.to_latex())
